# Result Plots

In [2]:
from utils.utils import *
import os
import json
import matplotlib.pyplot as plt
import math

In [3]:
TOPIC_ORDER = ['politics', 'general', 'covid', 'syria', 'islam', 'notredame', 'gossip'] # Topic order for plots
DATE_ORDER = ['2011-2015', '2016', '2017', '2019', '2020'] # Date order for plots
MODEL_ORDER = ['BERT', 'DeBERTa', 'RoBERTa', 'CNN', 'BiLSTM', 'Linear', 'NB', 'PA'] # Model order for plots

MODEL_COLORS = { # Colors for each model
    'BERT': 'tab:blue',
    'DeBERTa': 'tab:orange',
    'RoBERTa': 'tab:green',
    'CNN': 'tab:red',
    'BiLSTM': 'tab:purple',
    'Linear': 'tab:brown',
    'NB': 'tab:pink',
    'PA': 'tab:gray'
}

MODEL_MARKERS = { # Markers for each model
    'BERT': 'o',
    'DeBERTa': 's',
    'RoBERTa': '^',
    'CNN': 'D',
    'BiLSTM': 'v',
    'Linear': 'P',
    'NB': 'X',
    'PA': '*'
}

plt.rcParams.update({'font.size': 14})

#### Plot Functions

In [4]:
def load_data(type='topic', dir="", alpha=None, dim=None, lam=None):
    """
    Scan the RESULTS_DIR for model folders and load their JSON results, considering only the specified type.

    Args:
        type (str): Type of results to load ('topic' or 'date').
        dir (str): Suffix for results directory.
        alpha (float, optional): Alpha value for Distillation
        dim (int, optional): Dimension for Replay
        lam (float, optional): Lambda value for EWC

    Returns:
        dict: A dictionary with model names as keys and their loaded JSON data as values.
    """

    RESULTS_DIR = f'results_{dir}'
    model_data = {}

    # -------- build suffix dynamically --------
    # order matters: alpha then dim (as in your filenames)
    suffix_parts = []

    if alpha is not None:
        suffix_parts.append(str(alpha))

    if dim is not None:
        suffix_parts.append(str(dim))

    if lam is not None:
        suffix_parts.append(str(lam))

    # build suffix string
    extra_suffix = ""
    if suffix_parts:
        extra_suffix = "_" + "_".join(suffix_parts)

    for folder_name in os.listdir(RESULTS_DIR):
        folder_path = os.path.join(RESULTS_DIR, folder_name)
        
        # Considering only folders ending with the specified type
        if os.path.isdir(folder_path) and folder_name.endswith(f'_{type}'):
            parts = folder_name.split('_')
            
            # Get MODEL name from folder "results_{MODEL}_{type}"
            model_name = parts[1] 
            
            # JSON paths for Single / Cumulative / Full
            full_path = os.path.join(folder_path, f'results_full_{type}{extra_suffix}.json')
            cumulative_path = os.path.join(folder_path, f'results_cumulative_{type}{extra_suffix}.json')
            single_path = os.path.join(folder_path, f'results_single_{type}{extra_suffix}.json')
            
            data = {'full': None, 'cumulative': None, 'single': None}
            
            # Load data from Full Topic
            if os.path.exists(full_path):
                with open(full_path, 'r') as f:
                    data['full'] = json.load(f)
            
            # Load data from Cumulative Topic
            if os.path.exists(cumulative_path):
                with open(cumulative_path, 'r') as f:
                    data['cumulative'] = json.load(f)
            
            # Load data from Single Topic
            if os.path.exists(single_path):
                with open(single_path, 'r') as f:
                    data['single'] = json.load(f)
            
            # Store only if at least one data type is present
            if data['full'] or data['cumulative'] or data['single']:
                model_data[model_name] = data
            
            # Order models data according to MODEL_ORDER
            ordered_model_data = {k: model_data[k] for k in MODEL_ORDER if k in model_data}

    return ordered_model_data

In [19]:
data = load_data(type='topic', dir="finetuning")

In [20]:
print("=================================")
print("FULL TEST SET (Topic): F1 RESULTS")
print("=================================")

for model_name, d in data.items():
    #print(f"{model_name}: {d['full']}")
    print(f"{model_name}: {d['full'][TOPIC_ORDER[-1]]}") # only last topic for each model

FULL TEST SET (Topic): F1 RESULTS
BERT: 0.6747749301960299
DeBERTa: 0.7540822748205195
RoBERTa: 0.7498185230006515
CNN: 0.7308214674607046
BiLSTM: 0.625759184988451
Linear: 0.7412274017622653
NB: 0.7012035768448264
PA: 0.7665632898547037


In [ ]:
print("===================================")
print("SINGLE TEST SET (Topic): F1 RESULTS")
print("===================================\n")

for model_name, d in data.items():
    print(f"MODEL: {model_name}")
    #print(d['single'])
    #print(f"{d['single'][TOPIC_ORDER[-1]]}") # only last topic for each model
    #print("-" * (7 + len(model_name)))

SINGLE TEST SET (Topic): F1 RESULTS

MODEL: BERT
{'politics': 0.9334309837867284, 'general': 0.7559797579105044, 'covid': 0.9278752329439592, 'syria': 0.45490716180371354, 'islam': 0.9584325396825398, 'notredame': 0.8448353791143389, 'gossip': 0.6392190152801358}
-----------
MODEL: DeBERTa
{'politics': 0.9395041577730225, 'general': 0.7859600644949895, 'covid': 0.9274513772125201, 'syria': 0.5278697136853164, 'islam': 0.9583333333333334, 'notredame': 0.8630900403950048, 'gossip': 0.6744791666666667}
--------------
MODEL: RoBERTa
{'politics': 0.9417261473846785, 'general': 0.7889719617974629, 'covid': 0.9353357727954055, 'syria': 0.5565217391304348, 'islam': 0.9307208994708994, 'notredame': 0.8813447016756709, 'gossip': 0.7996794871794871}
--------------
MODEL: CNN
{'politics': 0.48639359487148953, 'general': 0.0016597339039069897, 'covid': 0.30761033418455086, 'syria': 0.296969696969697, 'islam': 0.23816365626710453, 'notredame': 0.2163332163332163, 'gossip': 0.33333333333333326}
-----

In [29]:
data = load_data(type='date', dir="finetuning")

In [30]:
print("================================")
print("FULL TEST SET (Date): F1 RESULTS")
print("================================\n")

for model_name, d in data.items():
    #print(f"{model_name}: {d['full']}")
    print(f"{model_name}: {d['full'][DATE_ORDER[-1]]}") # only last date for each model

FULL TEST SET (Date): F1 RESULTS

BERT: 0.7497599843886119
DeBERTa: 0.7485635379380123
RoBERTa: 0.7663085889911887
CNN: 0.711791556297837
BiLSTM: 0.3917229211769176
Linear: 0.7802984403769243
NB: 0.7645861654673534
PA: 0.771194108552689


In [10]:
print("==================================")
print("SINGLE TEST SET (Date): F1 RESULTS")
print("==================================\n")

for model_name, d in data.items():
    print(f"MODEL: {model_name}")
    #print(d['single'])
    print(f"{d['single'][DATE_ORDER[-1]]}") # only last date for each model
    print("-" * (7 + len(model_name)))

SINGLE TEST SET (Date): F1 RESULTS

MODEL: BERT
{'2011-2015': 0.34428316781257956, '2016': 0.6929460749035212, '2017': 0.6309758879788829, '2019': 0.7217637382450243, '2020': 0.8539591139788288}
-----------
MODEL: DeBERTa
{'2011-2015': 0.3893351537907232, '2016': 0.9331285453002186, '2017': 0.9924108688019022, '2019': 0.7684133601686793, '2020': 0.9758329179964754}
--------------
MODEL: RoBERTa
{'2011-2015': 0.4506006464938361, '2016': 0.9412736468000207, '2017': 0.9962771787885358, '2019': 0.7507154213036568, '2020': 0.9791516859257268}
--------------
MODEL: CNN
{'2011-2015': 0.37918593152844965, '2016': 0.8376703809039976, '2017': 0.9884359475498466, '2019': 0.8665812203956534, '2020': 0.9204055852969628}
----------
MODEL: BiLSTM
{'2011-2015': 0.3597742127153891, '2016': 0.8068818529255573, '2017': 0.9679189940087632, '2019': 0.6675675675675675, '2020': 0.9232131263428893}
-------------
MODEL: Linear
{'2011-2015': 0.5028638321000042, '2016': 0.8672744440454574, '2017': 0.955201905523

In [35]:
#lwf
alpha = ["03", "05"]

#replay
dim = [50, 100, 200]

#ewc
lam = ["300", "500", "1000"]

hybrid = [
    ("03", 50),
    ("03", 100),
    ("03", 200),
    ("05", 50),
    ("05", 100),
    ("05", 200)
]


for a, d in hybrid:
    #data = load_data(type='date', dir="replay", dim=d) # replay
    #data = load_data(type='date', dir="distillation", alpha=a) # distillation
    #data = load_data(type='date', dir="ewc", lam=l) # ewc
    data = load_data(type='date', dir="dist_rep", alpha=a, dim=d) # hybrid
    
    print("=================================")
    print("FULL TEST SET (Date): F1 RESULTS")
    print("=================================")
    #print(f"Replay (dim={d})")
    #print(f"Distillation (alpha={a})")
    #print(f"EWC (lambda={l})")
    print(f"Hybrid (alpha={a}, dim={d})")
    print("=================================")

    for model_name, d in data.items():
        #print(f"{model_name}: {d['full']}")
        print(f"{model_name}: {d['full'][DATE_ORDER[-1]]}") # only last task for each model
    
    print("\n")

FULL TEST SET (Date): F1 RESULTS
Hybrid (alpha=03, dim=50)
BERT: 0.8370504466163292
DeBERTa: 0.8043622135071276
RoBERTa: 0.8462092516579356
CNN: 0.6778712296587327
BiLSTM: 0.7064216756365919


FULL TEST SET (Date): F1 RESULTS
Hybrid (alpha=03, dim=100)
BERT: 0.8359078759294939
DeBERTa: 0.8472448462065554
RoBERTa: 0.879873670378755
CNN: 0.7544431603153284
BiLSTM: 0.7361888661091409


FULL TEST SET (Date): F1 RESULTS
Hybrid (alpha=03, dim=200)
BERT: 0.8841706947296736
DeBERTa: 0.8388587298259932
RoBERTa: 0.8631244004659503
CNN: 0.7452449804168205
BiLSTM: 0.7896601990270384


FULL TEST SET (Date): F1 RESULTS
Hybrid (alpha=05, dim=50)
BERT: 0.8244064716564016
DeBERTa: 0.8386927073387476
RoBERTa: 0.8513637501160994
CNN: 0.3601345065330373
BiLSTM: 0.17199869762264183


FULL TEST SET (Date): F1 RESULTS
Hybrid (alpha=05, dim=100)
BERT: 0.8644341709246608
DeBERTa: 0.8413794307552409
RoBERTa: 0.8756704883230345
CNN: 0.31454401923526315
BiLSTM: 0.17199869762264183


FULL TEST SET (Date): F1 RESUL

In [12]:
#lwf
#alpha = ["03", "05"]

#replay
dim = [50, 100, 200]

#ewc
#lam = ["300", "500", "1000"]

for d in dim:
    data = load_data(type='topic', dir="replay", dim=d)
    
    print("===================================")
    print("SINGLE TEST SET (Topic): F1 RESULTS")
    print("===================================")
    print(f"Distillation + Replay (dim={d}):")

    for model_name, d in data.items():
        print(f"MODEL: {model_name}")
        #print(d['single'])
        print(f"{d['single'][TOPIC_ORDER[-1]]}") # only last topic for each model
        print("-" * (7 + len(model_name)))
    
    print("\n")

SINGLE TEST SET (Topic): F1 RESULTS
Distillation + Replay (dim=50):
MODEL: BERT
{'politics': 0.8818413221772892, 'general': 0.6850257583322298, 'covid': 0.9478942675678287, 'syria': 0.4888608659100463, 'islam': 0.9375776937994131, 'notredame': 0.8645631462290972, 'gossip': 0.7373737373737375}
-----------
MODEL: DeBERTa
{'politics': 0.9391459496449747, 'general': 0.8162913006282482, 'covid': 0.929473099341302, 'syria': 0.5302809783707297, 'islam': 0.93036064332175, 'notredame': 0.8457317506463732, 'gossip': 0.8043455874781177}
--------------
MODEL: RoBERTa
{'politics': 0.9373462366839169, 'general': 0.7885993272351387, 'covid': 0.9372824806399221, 'syria': 0.5338135665894863, 'islam': 0.93036064332175, 'notredame': 0.8438100843164135, 'gossip': 0.7889659330720531}
--------------
MODEL: CNN
{'politics': 0.9174721203389404, 'general': 0.7972773232274731, 'covid': 0.8993130051358521, 'syria': 0.48607787274453945, 'islam': 0.875297619047619, 'notredame': 0.8168696865879964, 'gossip': 0.7144